In [ ]:
import sys; sys.path.insert(0, 'calibration')  # utils_calibrate* live in calibration/

In [ ]:
# %matplotlib inline
import numpy as np
import electric as electric
import utils_calibrate_fracrand
import importlib

importlib.reload(electric)
importlib.reload(utils_calibrate_fracrand)
from utils_calibrate_fracrand import (
    SimpleEFishAgent,
    plot_image_fish2,
    plot_line_fish2,
    get_vmin_vmax,
    run_self_image_experiment,
    run_direct_eod_sensing_experiment_no_food,
    run_direct_eod_sensing_experiment_no_food_fish2_grid,
    run_direct_eod_sensing_experiment_with_food,
    run_self_image_active_sense_other_fish_experiment,
    run_self_image_experiment_with_without_food
)
import matplotlib.pyplot as plt
import cfg
from cfg import AGENT_PARAMS, ENV_PARAMS, ELECTRIC_CONSTANTS, cm_to_m, m_to_cm
import matplotlib


print(matplotlib.get_backend())



## Set up values

In [ ]:
# Constants from your code
k_coulomb = ELECTRIC_CONSTANTS["k_coulomb"]  # Coulomb's constant
epsilon_0 = ELECTRIC_CONSTANTS["epsilon_0"]  # Vacuum permittivity (F/m)
epsilon0 = epsilon_0
chi = ELECTRIC_CONSTANTS["food_contrast"]
assert ELECTRIC_CONSTANTS["food_contrast"] == ELECTRIC_CONSTANTS["fish_contrast"]

mVcm_to_Vm = ELECTRIC_CONSTANTS["mVcm_to_Vm"]
print(f"mV/cm to V/m conversion factor: {mVcm_to_Vm:.2f}")
Vm_to_mVcm =ELECTRIC_CONSTANTS["Vm_to_mVcm"]
print(f"V/m to mV/cm conversion factor: {Vm_to_mVcm:.2f}")


fish_charge = AGENT_PARAMS["monopole_charges"][0]
print("fish_charge", fish_charge)



# Sensor thresholds: V/m (correct in cfg.py)
ampullary_sensor_min = AGENT_PARAMS["ampullary_sensor_min"]
ampullary_sensor_max = AGENT_PARAMS["ampullary_sensor_max"]
mormyromast_sensor_min = AGENT_PARAMS["mormyromast_sensor_min"] # REDUNDANT IN FRACRAND
mormyromast_sensor_max = AGENT_PARAMS["mormyromast_sensor_max"] # REDUNDANT IN FRACRAND
knollen_sensor_min = AGENT_PARAMS["knollen_sensor_min"] # No max for Knollen

print("\nSensor thresholds (V/m):")
print(f"Ampullary: {ampullary_sensor_min:.2e} -- {ampullary_sensor_max:.1e} V/m ({ampullary_sensor_min*Vm_to_mVcm:.1e} -- {ampullary_sensor_max*Vm_to_mVcm:.1e} mV/cm)")
print(f"Mormyromast: {mormyromast_sensor_min:.3f} -- {mormyromast_sensor_max:.3f} V/m ({mormyromast_sensor_min*Vm_to_mVcm:.3f} -- {mormyromast_sensor_max*Vm_to_mVcm:.3f} mV/cm)")
print(f"Knollen (min): {knollen_sensor_min:.3e} V/m ({knollen_sensor_min*Vm_to_mVcm:.3e} mV/cm)")

morm_max_multiplier = AGENT_PARAMS["morm_max_multiplier"]
morm_min_multiplier = AGENT_PARAMS["morm_min_multiplier"]
print(f"mormyromast min--max multipliers: {morm_min_multiplier:.2e}--{morm_max_multiplier:.2f}")


## Mormyromast Self-EOD with Food

In [ ]:
importlib.reload(utils_calibrate_fracrand)
from utils_calibrate_fracrand import (
    run_self_image_experiment,
)
criterion = ["max", "min"][0]
summary_readings, fish, all_readings, baseline = run_self_image_experiment(
    fish_orientation=np.pi / 2,
    num_points=50,
    buffer_cm=12,
    arena_size=(100, 100),
    fish_charge=fish_charge,
    do_self_induced=False,
    do_baseline_subtraction=True,
    criterion=criterion,
    no_food=False
)   
# clip to 1e-25
summary_readings[:, 2] = np.clip(
    summary_readings[:, 2], 1e-25, None
)
print("readings", summary_readings[:, 2])
baseline_max = np.max(np.abs(baseline))
contour_max = baseline_max * (1 + morm_max_multiplier)
contour_min = baseline_max * morm_min_multiplier
print("baseline_max", baseline_max)
print("np.log10(contour_min)", np.log10(contour_min))
print("np.log10(contour_max)", np.log10(contour_max))

contour_levels_morm_self_image_food_fish = np.log10([contour_min, contour_max])

print("contour_levels_morm_self_image_food_fish", contour_levels_morm_self_image_food_fish)
plot_image_fish2(
    readings=summary_readings,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_morm_self_image_food_fish,  # morm min and max
    criterion=criterion,
)


## Mormyromast Self-EOD with Other Fish

In [ ]:
importlib.reload(utils_calibrate_fracrand)
from utils_calibrate_fracrand import run_self_image_active_sense_other_fish_experiment


readings_self, fish = run_self_image_active_sense_other_fish_experiment(
    fish_orientation=np.pi / 2,
    other_fish_orientation=np.pi / 2,
    num_points=50,
    buffer=12,  # cm buffer around fish body
    fish_charge=fish_charge,  # C
    do_self_induced=False,
    do_baseline_subtraction=True,
    criterion="max",
)

print("contour_levels_morm_self_image_food_fish", contour_levels_morm_self_image_food_fish)
plot_image_fish2(
    readings=readings_self,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_morm_self_image_food_fish,
    criterion="max",
)

## Ampullary Sensing Food

In [ ]:
importlib.reload(utils_calibrate_fracrand)
importlib.reload(cfg)
from utils_calibrate_fracrand import run_sense_food_ampullary


max_readings_self, fish = run_sense_food_ampullary(
    fish_orientation=np.pi / 2,
    num_points=50,
)

print("fish_charge", fish_charge)

ampullary_sensor_min = AGENT_PARAMS["ampullary_sensor_min"]  # V/m
ampullary_sensor_max = AGENT_PARAMS["ampullary_sensor_max"]  # V/m
print("[ampullary_sensor_min, ampullary_sensor_max]", [ampullary_sensor_min, ampullary_sensor_max])
contour_levels_ampullary_self_image = np.log10(
    [ampullary_sensor_min, ampullary_sensor_max]
)  # V/m


plot_image_fish2(
    readings=max_readings_self,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_ampullary_self_image,
)

## Ampullary Sensing Other Fish

In [ ]:
importlib.reload(utils_calibrate_fracrand)
importlib.reload(cfg)
from utils_calibrate_fracrand import run_sense_other_fish_ampullary


max_readings_self, fish = run_sense_other_fish_ampullary(
    fish_orientation=np.pi / 2,
    other_fish_orientation=np.pi / 2,
    num_points=50,
)

plot_image_fish2(
    readings=max_readings_self,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_ampullary_self_image,
)

## Knollen Sensing Cons-EOD 
* Directly/passively sensing cons-EOD (not "exactly" cons-image)
* Move Fish 1 around and sense on Fish 2 

In [ ]:
importlib.reload(utils_calibrate_fracrand)
importlib.reload(electric)
import arena
from utils_calibrate_fracrand import (
    run_direct_eod_sensing_experiment_no_food,
)

knollen_sensor_min = AGENT_PARAMS["knollen_sensor_min"]  # V/m
print("knollen_sensor_min", knollen_sensor_min)
print("fish_charge", fish_charge)

contour_levels_knoll_sense_cons_eod = np.log10([knollen_sensor_min])  # V/m
print(f"Contour levels: {contour_levels_knoll_sense_cons_eod}")
max_readings_direct, fish2 = run_direct_eod_sensing_experiment_no_food(
    num_points=50,
    do_self_induced=False,
    do_induced=True,
    arena_size=(300, 300),
    buffer_cm=150,
    fish_charge=fish_charge,
    criterion='max'
)

plot_image_fish2(
    readings=max_readings_direct,
    fish1=None,
    fish2=fish2,
    type="cons",
    contour_levels=contour_levels_knoll_sense_cons_eod,

)